# MESA Datagen

Datagen passes each of these synthetic documents to (the same) foundation model, along with a custom schema containing target fields of interest (derived from domain knowledge).
The model is prompted to standardise each document to the schema, creating a set of pairs illustrating the standardisation process.

## Pre-requisites

- AWS credentials

## Overview

Datagen sits alongside finetune as the 'build' components of MESA (below). As with docsynth, a key part of the datagen process is to derive a suitable prompt, in this case one that can elicit a comprehensive standardised version of each synthetic document. Unlike docsynth, prompts aren't derived dynamically but hardcoded to capture domain insight, and thus form versionable assets that are fed into the datagen component. The schema itself is also a key asset, capturing key domain knowledge of the types of data (often biomarkers) likely contained in the free-text being processed. With these two assets, datagen effectively becomes middleware, interfacing with external services to derive document:schema pairs, and parse, check and reformat them as required, prior to them being used for finetuning. 

<p align="center">
  <img src="https://londonaicentre.github.io/MESA-Build/repo.png" width="700" alt="Flowchart of the document generation pipeline">
</p>

As with docsynth, new prompts and schemas can be developed for new domains. Again, as with docsynth, this library also contains prompts and schemas for existing domains. In keeping with the running example, we will work with an oncology schema and associated prompts in this workbook.

## Steps

1. Import the relevant datagen packages: we want to remind ourselves what our Docsynth outputs are called, use AWS batch processing again, and then want to upload our data.

In [ ]:
from datagen import BedrockBatchGenerator, DocumentLoader, TrainingDataUploader

2. Import the relevant asset packages: we want to use 'oncoschema', which includes the prompts and the schema itself. We also want to instantiate the prompt wrapper class. 

In [ ]:
from oncoschema.prompt_builder import PromptBuilder
from oncoschema import Schema
prompt_builder: PromptBuilder = PromptBuilder()

3. Import some utility classes to help us work with the library.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

4. The last step during docsynth uploaded a set of synthetic documents to AWS. We can use a tool from the datagen library to identify what the name of this upload is.

In [ ]:
DocumentLoader().list_available_document_batches()

5. We now want to create a reusable datagen batch generator object. To do so, we supply the prompt to use (system prompt) from our prompt assets, the schema from those same assets, the name of our schema, the model we want to use, and the name of the document batch we generated from docsynth. Creating this object will automatically download this batch for us.

In [ ]:
generator: BedrockBatchGenerator = BedrockBatchGenerator(
    system_prompt=prompt_builder.build_datagen_prompt(),
    schema=Schema,
    schema_name="oncoschema",
    model_name="sonnet4",
    document_batches=["test_batch-2026-07-05-001.tar.gz"],
)

6. Run the following code to start batch generation of document:schema pairs, with AWS Bedrock batch processing leveraging the credentials we already have in place.

❗🪙 Batch inference can be an expensive process. If you would prefer to skip this step, instead run `mv .job_id.example.json .job_id.json`.

In [ ]:
generator.generate_via_batch(100, os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

7. (Optional) Inspect `anthropic_batch_job.jsonl` to see the prompts we are sending to AWS Bedrock (Claude Sonnet) to generate our finetuning pairs.

8. Because this a batch job, we may need to wait before we can process the outputs. We can check the status of the job by running the command below.

In [ ]:
generator.check_batch_output_status(os.environ["BUCKET"])

9. Once the above check confirms our batch output is ready, we can download it, parse it and output it in a format for the rest of the pipeline. You may notice some validation failures reported; this is understandable when generating large amounts of training data, but you'll also notice that we catch and exclude failing pairs so that they don't pollute our training data.

In [ ]:
generator.extract_batch_output(os.environ["BUCKET"])

10. (Optional) Inspect `data/trainingdata` to see the format of the training pairs we have generated.

11. Passing much of the same data we have passed previously, we now upload the training data for use by the next component.

In [ ]:
TrainingDataUploader.upload(
    schema=Schema,
    schema_name="oncoschema",
    system_prompt=prompt_builder.build_main_prompt(),
    short_description="oncotest-batch",
    long_description="Training data for OncoTest",
    input_folder=Path("./data/trainingdata"),
)